In [3]:
# import module
import pandas as pd
import os
import numpy as np

# 1. LOAD DATASET

In [7]:
# Path ke file (sesuaikan dengan folder Anda)
path_clean = r'D:\Dokumen\Kode_00B\SatriaData\Clean Data'
path_raw = r'D:\Dokumen\Kode_00B\SatriaData\Raw Data'

# 1a. Data utama (IPP + IPLM) yang sudah digabung sebelumnya
df_utama = pd.read_csv(os.path.join(path_clean, 'data_final.csv'))
print(f"1. Data utama: {df_utama.shape[0]} baris, {df_utama.shape[1]} kolom")
print(f"   Tahun tersedia: {sorted(df_utama['tahun'].unique())}")
print(f"   Provinsi: {df_utama['nama_provinsi'].nunique()}")

# 1b. Indeks Daya Saing Digital
df_digital = pd.read_csv(os.path.join(path_raw, 'vertikalkementerian-2-od_20431_indeks_daya_saing_digital__prov_di_indonesia_v6_data.csv'))
print(f"\n2. Data Daya Saing Digital: {df_digital.shape[0]} baris, {df_digital.shape[1]} kolom")
print(f"   Tahun tersedia: {sorted(df_digital['tahun'].unique())}")
print(f"   Provinsi: {df_digital['nama_provinsi'].nunique()}")

# 1c. IMDI (Indeks Masyarakat Digital Indonesia) - baca hanya 4 kolom pertama
df_imdi_list = []
tahun_imdi = [2022, 2023, 2024]

for tahun in tahun_imdi:
    file_name = f'Data Indeks Masyarakat Digital Indonesia Provinsi {tahun}.xlsx'
    file_path = os.path.join(path_raw, file_name)
    
    # Baca file Excel, ambil kolom A sampai D (Tahun, Kode, Provinsi, Indeks)
    df_temp = pd.read_excel(file_path, sheet_name='Sheet1', header=0, usecols='A:D')
    
    # Rename langsung berdasarkan posisi kolom (4 kolom pertama)
    df_temp.columns = ['tahun', 'kode_provinsi', 'nama_provinsi', 'indeks_masyarakat_digital']
    
    # Hapus baris dengan nilai kosong pada kolom indeks
    df_temp = df_temp.dropna(subset=['indeks_masyarakat_digital'])
    
    # Konversi tipe data
    df_temp['tahun'] = df_temp['tahun'].astype(int)
    df_temp['kode_provinsi'] = df_temp['kode_provinsi'].astype(int)
    
    df_imdi_list.append(df_temp)
    print(f"   - IMDI {tahun}: {len(df_temp)} provinsi")

# Gabungkan semua tahun IMDI
df_imdi_all = pd.concat(df_imdi_list, ignore_index=True)
print(f"\n3. Data IMDI total: {df_imdi_all.shape[0]} baris, {df_imdi_all.shape[1]} kolom")
print(f"   Tahun tersedia: {sorted(df_imdi_all['tahun'].unique())}")
print(f"   Provinsi: {df_imdi_all['nama_provinsi'].nunique()}")

1. Data utama: 135 baris, 5 kolom
   Tahun tersedia: [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
   Provinsi: 34

2. Data Daya Saing Digital: 174 baris, 6 kolom
   Tahun tersedia: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
   Provinsi: 38
   - IMDI 2022: 34 provinsi
   - IMDI 2023: 37 provinsi
   - IMDI 2024: 38 provinsi

3. Data IMDI total: 109 baris, 4 kolom
   Tahun tersedia: [np.int64(2022), np.int64(2023), np.int64(2024)]
   Provinsi: 38


# 2. GABUNGKAN SEMUA DATASET

In [8]:
# 2a. Gabungkan data utama dengan Daya Saing Digital
df_merge1 = pd.merge(
    df_utama,
    df_digital[['kode_provinsi', 'nama_provinsi', 'tahun', 'indeks_daya_saing_digital']],
    on=['kode_provinsi', 'nama_provinsi', 'tahun'],
    how='left'  # left join agar semua data utama tetap ada
)
print(f"\n4. Gabung + Daya Saing: {df_merge1.shape[0]} baris, {df_merge1.shape[1]} kolom")
print(f"   Missing Daya Saing Digital: {df_merge1['indeks_daya_saing_digital'].isna().sum()}")

# 2b. Gabungkan dengan IMDI
df_final = pd.merge(
    df_merge1,
    df_imdi_all[['kode_provinsi', 'nama_provinsi', 'tahun', 'indeks_masyarakat_digital']],
    on=['kode_provinsi', 'nama_provinsi', 'tahun'],
    how='left'
)
print(f"\n5. Gabung + IMDI: {df_final.shape[0]} baris, {df_final.shape[1]} kolom")
print(f"   Missing IMDI: {df_final['indeks_masyarakat_digital'].isna().sum()}")


4. Gabung + Daya Saing: 135 baris, 6 kolom
   Missing Daya Saing Digital: 0

5. Gabung + IMDI: 135 baris, 7 kolom
   Missing IMDI: 35


# 3. CLEANING & FILTER

In [9]:
# Filter hanya tahun 2022-2024 (karena IMDI baru mulai 2022)
df_final = df_final[df_final['tahun'].between(2022, 2024)].copy()
print(f"\n6. Setelah filter 2022-2024: {df_final.shape[0]} baris, {df_final.shape[1]} kolom")

# Cek missing values
print("\n7. Missing values per kolom:")
print(df_final.isnull().sum())

# Cek apakah ada provinsi yang hilang di salah satu variabel
print("\n8. Cek cakupan data per tahun:")
print(df_final.groupby('tahun').agg({
    'indeks_pelayanan_publik': 'count',
    'indeks_pembangunan_literasi': 'count',
    'indeks_daya_saing_digital': 'count',
    'indeks_masyarakat_digital': 'count'
}))


6. Setelah filter 2022-2024: 101 baris, 7 kolom

7. Missing values per kolom:
kode_provinsi                  0
nama_provinsi                  0
tahun                          0
indeks_pelayanan_publik        0
indeks_pembangunan_literasi    0
indeks_daya_saing_digital      0
indeks_masyarakat_digital      1
dtype: int64

8. Cek cakupan data per tahun:
       indeks_pelayanan_publik  indeks_pembangunan_literasi  \
tahun                                                         
2022                        34                           34   
2023                        34                           34   
2024                        33                           33   

       indeks_daya_saing_digital  indeks_masyarakat_digital  
tahun                                                        
2022                          34                         34  
2023                          34                         33  
2024                          33                         33  


# 4. SIMPAN DATA FINAL

In [10]:
# Simpan ke Clean Data
output_path = os.path.join(path_clean, 'data_panel_lengkap.csv')
df_final.to_csv(output_path, index=False)
print(f"\n9. Data lengkap disimpan di: {output_path}")

# Tampilkan 10 baris pertama
print("\n10. Contoh data:")
print(df_final.head(10))


9. Data lengkap disimpan di: D:\Dokumen\Kode_00B\SatriaData\Clean Data\data_panel_lengkap.csv

10. Contoh data:
    kode_provinsi nama_provinsi  tahun  indeks_pelayanan_publik  \
1              11          ACEH   2022                     4.01   
2              11          ACEH   2023                     4.34   
3              11          ACEH   2024                     4.45   
5              51          BALI   2022                     4.00   
6              51          BALI   2023                     4.21   
7              51          BALI   2024                     4.41   
9              36        BANTEN   2022                     3.99   
10             36        BANTEN   2023                     3.74   
11             36        BANTEN   2024                     3.73   
13             17      BENGKULU   2022                     4.40   

    indeks_pembangunan_literasi  indeks_daya_saing_digital  \
1                         58.46                       32.7   
2                        

# 5. STATISTIK DESKRIPTIF AWAL

In [11]:
print("\n" + "="*60)
print("STATISTIK DESKRIPTIF VARIABEL (2022-2024)")
print("="*60)
print(df_final[['indeks_pelayanan_publik', 
                'indeks_pembangunan_literasi',
                'indeks_daya_saing_digital',
                'indeks_masyarakat_digital']].describe())

# Korelasi
print("\n" + "="*60)
print("KORELASI ANTAR VARIABEL")
print("="*60)
print(df_final[['indeks_pelayanan_publik', 
                'indeks_pembangunan_literasi',
                'indeks_daya_saing_digital',
                'indeks_masyarakat_digital']].corr())


STATISTIK DESKRIPTIF VARIABEL (2022-2024)
       indeks_pelayanan_publik  indeks_pembangunan_literasi  \
count               101.000000                   101.000000   
mean                  3.891287                    67.045743   
std                   0.675600                    10.389770   
min                   1.110000                    20.020000   
25%                   3.600000                    62.520000   
50%                   4.040000                    66.880000   
75%                   4.340000                    72.730000   
max                   4.760000                    88.240000   

       indeks_daya_saing_digital  indeks_masyarakat_digital  
count                 101.000000                 100.000000  
mean                   40.317822                  41.730900  
std                     9.480099                   4.714215  
min                    24.900000                  20.901950  
25%                    34.200000                  39.716368  
50%              